# 01 — Tensor bridge and dataset export

How `ModelPart` data becomes a `torch.Tensor` (and comes back), and how to export
simulation fields as an `.npz` training dataset consumable by a `torch.utils.data.DataLoader`.

> **Prerequisites**: a compiled Kratos with `PhysicsNeMoApplication` enabled and, on the
> environment running this notebook:
>
> ```bash
> export PYTHONPATH=/path/to/Kratos/bin/Release
> export LD_LIBRARY_PATH=/path/to/Kratos/bin/Release/libs:$LD_LIBRARY_PATH
> pip install torch nvidia-physicsnemo   # optional runtime dependencies of the application
> ```
>
> Everything below runs on CPU in a few seconds; outputs are written to `./output/`.

In [1]:
from pathlib import Path

import numpy
import torch

import KratosMultiphysics as Kratos
from KratosMultiphysics.PhysicsNeMoApplication.bridges import torch_bridge
from KratosMultiphysics.PhysicsNeMoApplication.processes.export.dataset_export_process import DatasetExportProcess
from KratosMultiphysics.PhysicsNeMoApplication.training.torch_dataset import CreateNpzDataset

torch.manual_seed(0)
Path("output").mkdir(exist_ok=True)

## A small model part

Ten nodes on a line, carrying a historical `VELOCITY` (vector) and `PRESSURE` (scalar).

In [2]:
model = Kratos.Model()
model_part = model.CreateModelPart("Main")
model_part.AddNodalSolutionStepVariable(Kratos.VELOCITY)
model_part.AddNodalSolutionStepVariable(Kratos.PRESSURE)
for i in range(10):
    node = model_part.CreateNewNode(i + 1, i / 9.0, 0.0, 0.0)
    node.SetSolutionStepValue(Kratos.VELOCITY, [node.X, 2.0 * node.X, 0.0])
    node.SetSolutionStepValue(Kratos.PRESSURE, node.X**2)
print(f"{model_part.NumberOfNodes()} nodes")

10 nodes


## ModelPart → torch, zero-copy

`KratosTensorToTorch` wraps the tensor adaptor's staging buffer without copying:
mutating the torch tensor mutates the adaptor's `.data` view. (`CollectData()` /
`StoreData()` move data across the last hop to the actual — non-contiguous — Kratos
entity storage.)

In [3]:
ta = Kratos.TensorAdaptors.HistoricalVariableTensorAdaptor(model_part.Nodes, Kratos.VELOCITY)
ta.CollectData()
velocity = torch_bridge.KratosTensorToTorch(ta)
print("shape:", tuple(velocity.shape), "dtype:", velocity.dtype)

velocity[0, 0] = -42.0  # zero-copy: this writes into the adaptor's buffer
print("adaptor sees the mutation:", ta.data[0, 0])

shape: (10, 3) dtype: torch.float64
adaptor sees the mutation: -42.0


## torch → ModelPart

`TorchToKratosTensor` copies the values in (the tensor may live on a GPU or in an
autograd graph) and calls `StoreData()` so they reach the nodes.

In [4]:
torch_bridge.TorchToKratosTensor(2.0 * velocity, ta)
print("node 1 VELOCITY_X after write-back:", model_part.GetNode(1).GetSolutionStepValue(Kratos.VELOCITY_X))

node 1 VELOCITY_X after write-back: -84.0


## Exporting a training dataset

`DatasetExportProcess` writes one `.npz` per sampled step — keys are
`<VARIABLE>__<location>` plus `TIME`/`STEP`. It has **no ML dependency at all**, so it
runs on any Kratos installation (e.g. inside an HPC job).

In [5]:
settings = Kratos.Parameters("""{
    "model_part_name" : "Main",
    "list_of_fields"  : [
        { "variable_name" : "VELOCITY", "data_location" : "node_historical" },
        { "variable_name" : "PRESSURE", "data_location" : "node_historical" }
    ],
    "output_path"     : "output/dataset",
    "output_interval" : 1
}""")
export = DatasetExportProcess(model, settings)
export.ExecuteInitialize()

for step in range(1, 6):
    model_part.ProcessInfo[Kratos.STEP] = step
    model_part.ProcessInfo[Kratos.TIME] = 0.1 * step
    for node in model_part.Nodes:  # stand-in for an actual solve
        node.SetSolutionStepValue(Kratos.PRESSURE, node.X**2 + 0.1 * step)
    export.ExecuteFinalizeSolutionStep()

sorted(p.name for p in Path("output/dataset").glob("*.npz"))

## Training on the export

`CreateNpzDataset` exposes the directory as a `torch.utils.data.Dataset`, with the same
per-entity `(n, width)` concatenated layout `InferenceProcess` uses at deployment time.

In [6]:
dataset = CreateNpzDataset("output/dataset",
                           input_keys=["VELOCITY__node_historical"],
                           output_keys=["PRESSURE__node_historical"])
loader = torch.utils.data.DataLoader(dataset, batch_size=2)
for inputs, targets in loader:
    print("batch:", tuple(inputs.shape), "->", tuple(targets.shape))

batch: (2, 10, 3) -> (2, 10, 1)
batch: (2, 10, 3) -> (2, 10, 1)
batch: (1, 10, 3) -> (1, 10, 1)
